## Text Extraction, Cleaning, and Rephrasing from multiple books

In [1]:
import pandas as pd
import traceback
import json
from PyPDF2 import PdfReader
import os

from bson import ObjectId
from motor.motor_asyncio import AsyncIOMotorClient
from langchain_openai import ChatOpenAI
from langchain import PromptTemplate, LLMChain

from latext_Prompts import (
EXTRACTION_SYSTEM,
EXTRACTION_USER,
EXTRACTION_OUTPUT_FORMAT,
MATHEMATICS_EXTRACTION_OUTPUT_EXAMPLE,
BIOLOGY_EXTRACTION_OUTPUT_EXAMPLE,
PHYSICS_EXTRACTION_OUTPUT_EXAMPLE,
CHEMISTRY_EXTRACTION_OUTPUT_EXAMPLE,
EMPTY_EXTRACTION_OUTPUT_FORMAT,
CLEAN_REPHRASE_USER,
CLEAN_REPHRASE_SYSTEM,
REPHRASE_OUTPUT_FORMAT,
REPHRASE_OUTPUT_EXAMPLE,
BIOLOGY_SOLUTION_SYSTEM_PROMPT,
CHEMISTRY_SOLUTION_SYSTEM_PROMPT,
MATHEMATICS_SOLUTION_SYSTEM_PROMPT,
PHYSICS_SOLUTION_SYSTEM_PROMPT,
DISTRACTOR_SYSTEM_PROMPT
)

from config import (
    OPENAI_API_KEY,
    MONGO_URL,
    DB_NAME
)

In [2]:
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_API_KEY"] = "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"
# os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
# os.environ["LANGCHAIN_PROJECT"] = "Questions Extractor"

# Pipeline

In [3]:
import json
def parse_Response(response):
    if isinstance(response, str):
        # print(response)
        start_index = response.find("{")
        end_index = response.rfind("}")
        if start_index != -1 and end_index != -1:
            valid_json_content = response[start_index : end_index + 1]
            try:
                JSON_response = json.loads(valid_json_content.replace("\n", "").replace("\(", "").replace("\)", ""))
                # append_list_to_file(JSON_response)
                return JSON_response
            except json.JSONDecodeError as e:
                print(f"Error decoding JSON response: {e.__class__.__name__} - {e}\n\n Still trying to work on particular exceptions ...")
                if str(e).startswith("Extra data"):
                    json_parts = valid_json_content.split('}\n{')
                    if json_parts:
                        first_json = json_parts[0] + '}'
                        return parse_Response(first_json)  
                elif str(e).startswith("Invalid \escape"):
                    print("Before removing \\ issue: ", valid_json_content)
                    str1 = valid_json_content.replace("\\\\\\\\\\", "fvback")
                    str1 = str1.replace("\\\\\\\\", "frback")
                    str1 = str1.replace("\\\\\\", "trlback")
                    str1 = str1.replace("\\\\", "dblback")
                    str1 = str1.replace("\\", "\\\\")
                    str1 = str1.replace("fvback","\\\\\\\\\\")
                    str1 = str1.replace("frback", "\\\\\\\\")
                    str1 = str1.replace("trlback","\\\\\\")
                    strfinal = str1.replace("dblback","\\\\")
                    print("After removing \\ issue: ", strfinal)
                    JSON_response = json.loads(strfinal.replace("\n", ""))
                    return JSON_response
                else:
                    print("Actual Content: ", valid_json_content)
        else:
            print("No valid JSON content found in the response.")
        # time.sleep(50)
    elif isinstance(response, dict):
        return response
    else:
        print("No response message found", type(response))

<>:10: SyntaxWarning: invalid escape sequence '\('
<>:10: SyntaxWarning: invalid escape sequence '\)'
<>:20: SyntaxWarning: invalid escape sequence '\e'
<>:10: SyntaxWarning: invalid escape sequence '\('
<>:10: SyntaxWarning: invalid escape sequence '\)'
<>:20: SyntaxWarning: invalid escape sequence '\e'
C:\Users\aniket singh\AppData\Local\Temp\ipykernel_12884\3339088456.py:10: SyntaxWarning: invalid escape sequence '\('
  JSON_response = json.loads(valid_json_content.replace("\n", "").replace("\(", "").replace("\)", ""))
C:\Users\aniket singh\AppData\Local\Temp\ipykernel_12884\3339088456.py:10: SyntaxWarning: invalid escape sequence '\)'
  JSON_response = json.loads(valid_json_content.replace("\n", "").replace("\(", "").replace("\)", ""))
C:\Users\aniket singh\AppData\Local\Temp\ipykernel_12884\3339088456.py:20: SyntaxWarning: invalid escape sequence '\e'
  elif str(e).startswith("Invalid \escape"):


In [4]:
# All question collection

def collect_questions_from_chapter_with_Langchain(subject, chapter_text, chapter_name, grade):
    try:
        # initialize the ChapOpenAI object
        llm = ChatOpenAI(
            model_name="gpt-4o",
            api_key= OPENAI_API_KEY,
            temperature=0.2,
        )
        match subject:
            case "mathematics":
                examples=MATHEMATICS_EXTRACTION_OUTPUT_EXAMPLE
            case "biology":
                examples=BIOLOGY_EXTRACTION_OUTPUT_EXAMPLE
            case "physics":
                examples =PHYSICS_EXTRACTION_OUTPUT_EXAMPLE
            case "chemistry":
                examples=CHEMISTRY_EXTRACTION_OUTPUT_EXAMPLE

        #formatting prompts
        system = EXTRACTION_SYSTEM.format("",grade = grade, subject = subject,EMPTY_EXTRACTION_OUTPUT_FORMAT = EMPTY_EXTRACTION_OUTPUT_FORMAT, out_format = EXTRACTION_OUTPUT_FORMAT, out_example = examples)
        user = EXTRACTION_USER.format(chapter_name = chapter_name, chapter_text = chapter_text)

        #generating response from api call
        response = llm.invoke(
            [
                ("system", system),
                ("human", user)
            ],
        )
        return response
    except Exception as e:
        print("Error with in extraction: ", type(e).__name__, "–", e, "\n", traceback.format_exc())
        return None

In [5]:
def clean_rephrase_question(question_text, topic_list):
    try:
        llm = ChatOpenAI(
            model_name="gpt-4o",
            api_key= OPENAI_API_KEY,
            temperature=0.2,
            # verbose = True
        )
        # client = OpenAI(base_url='https://api.deepseek.com/v1', api_key='sk-c25c646d6e3a41c3801a44766a4baa48')



        #formatting prompts
        system = CLEAN_REPHRASE_SYSTEM.format("",out_example = REPHRASE_OUTPUT_EXAMPLE, out_format = REPHRASE_OUTPUT_FORMAT)
        user = CLEAN_REPHRASE_USER.format(question_text = question_text, topic_list = topic_list)

        # #generating response from api call
        response = llm.invoke(
            [
                ("system", system),
                ("human", user)
            ]
        )
        # response = client.chat.completions.create(model='deepseek-reasoner',
        #                             messages= [
        #                                 {'role':'system', 'content': system },
        #                                 {'role':'user', 'content': user },
        #                                         ],
        #                             # response_format={'type': 'json_object', 'schema': schema},
        #                             temperature=0.2,
        #                             )
        return response

    except Exception as e:
        print("Error cleaning text: ", type(e).__name__, "–", e, "\n", traceback.format_exc())
        return question_text  # Return original text in case of an error

In [6]:
import os
import traceback
from pathlib import Path
import zipfile
import tempfile
import shutil

def read_data_from_latex(publication: str, chapter_name: str , grade:str) -> str:
    """
    Read LaTeX content from a specified publication and chapter zip file.
    
    Args:
        publication (str): Name of the publication (e.g., 'mtg')
        chapter_name (str): Name of the chapter (e.g., 'Life Processes')
    
    Returns:
        str: Content of the LaTeX file if successful, empty string if failed
    """
    try:
        # Construct the zip file path
        zip_path = os.path.join(r"textbooks", publication, f"{chapter_name}_{grade}.zip")
        
        # Verify zip file exists
        if not os.path.exists(zip_path):
            raise FileNotFoundError(f"Zip file not found: {zip_path}")
        
        # Create a temporary directory to extract files
        with tempfile.TemporaryDirectory() as temp_dir:
            # Extract the zip file
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(temp_dir)
            
            # Find directories starting with 2025
            content_dirs = [d for d in os.listdir(temp_dir) 
                          if os.path.isdir(os.path.join(temp_dir, d)) and 
                          d.startswith('2025')]
            
            if not content_dirs:
                raise FileNotFoundError(f"No content directories found in zip file")
                
            # Get the latest directory
            latest_dir = sorted(content_dirs)[-1]
            dir_path = os.path.join(temp_dir, latest_dir)
            
            # Find the .tex file in the directory
            tex_files = [f for f in os.listdir(dir_path) if f.endswith('.tex')]
                    
            if not tex_files:
                raise FileNotFoundError(f"No .tex file found in {dir_path}")
                
            # Full path to the tex file
            latex_path = os.path.join(dir_path, tex_files[0])
            
            # Read the content
            with open(latex_path, 'r', encoding='utf-8') as file:
                content = file.read()
                
            print(f"Successfully processed LaTeX file from zip: {latex_path}")
            return content

    except Exception as e:
        print(
            "Failed to process the LaTeX file. Exception Occurred:",
            type(e).__name__,
            "–",
            e,
            "\n",
            traceback.format_exc()
        )
        return ""

In [7]:
def extract_questions(subject: str, chapter_texts: list, chapter_name: str, grade: str) -> list:
    try:
        # Validate inputs
        if not isinstance(chapter_texts, list):
            raise ValueError("chapter_texts must be a list of strings")
            
        if not chapter_texts:
            print(f"Warning: Empty chapter_texts for chapter {chapter_name}")
            return []
            
        all_questions = []
        
        # Process lines in chunks of 500
        chunk_size = 250
        for chunk_index in range(0, len(chapter_texts), chunk_size):
            try:
                # Get current chunk of lines
                chunk = chapter_texts[chunk_index:chunk_index + chunk_size]
                
                # Skip empty chunks
                if not chunk:
                    continue
                    
                print(f"Processing chunk {chunk_index//chunk_size + 1}/{-(-len(chapter_texts)//chunk_size)}")
                
                # Use the original collection function
                questions = collect_questions_from_chapter_with_Langchain(
                    subject=subject,
                    chapter_text="\n".join(chunk),
                    chapter_name=chapter_name,
                    grade=grade
                )
                
                # Parse and store valid questions
                if questions and questions.content:
                    parsed = parse_Response(questions.content)
                    if parsed and "questions" in parsed:
                        questions_list = parsed["questions"]
                        if questions_list:
                            all_questions.extend(questions_list)
                            print(f"Extracted {len(questions_list)} questions from chunk {chunk_index//chunk_size + 1}")
                
            except Exception as e:
                print(f"Error processing chunk {chunk_index//chunk_size + 1}: {str(e)}")
                continue
        
        print(f"Total questions extracted: {len(all_questions)}")
        return all_questions

    except Exception as e:
        print(f"Error in extract_questions: {str(e)}")
        return []

In [8]:
def rephrase_questions(conf_data: dict, all_questions: list, topics: list) -> dict:
    print("Starting the cleanup and rephrasing process...")
    try:
        # Validate inputs
        if not conf_data or "chapter_name" not in conf_data:
            raise ValueError("Missing chapter_name in configuration")
            
        if not topics:
            print("\n\nNo topics found in the data.")
            raise ValueError("No topics found in the data")
            
        chapter = conf_data["chapter_name"]
        clean_rephrased_questions = {}
        chapterwise_formatted_questions = []
        
        if not all_questions:
            raise ValueError(f"No questions found for chapter {chapter}")
            
        print(f"\nWorking on chapter {chapter}")
        print(f"Total questions to process: {len(all_questions)}")
        
        # Process questions in batches of 5
        batch_size = 5
        for batch_start in range(0, len(all_questions), batch_size):
            try:
                # Get current batch of questions
                question_batch = all_questions[batch_start:batch_start + batch_size]
                print(f"\nProcessing batch {batch_start//batch_size + 1}/{-(-len(all_questions)//batch_size)}")
                
                # Clean and rephrase the batch
                cleaned_rephrased_questions = clean_rephrase_question(question_batch, topics)
                print("Raw response:", cleaned_rephrased_questions)
                
                # Parse the cleaned response
                if cleaned_rephrased_questions and cleaned_rephrased_questions.content:
                    formatted_questions = parse_Response(cleaned_rephrased_questions.content)
                    if formatted_questions and "questions" in formatted_questions:
                        chapterwise_formatted_questions.extend(formatted_questions["questions"])
                        print(f"Successfully processed {len(formatted_questions['questions'])} questions in batch")
                    else:
                        print(f"Warning: No valid questions found in batch {batch_start//batch_size + 1}")
                
                print("-" * 100)
                
            except Exception as batch_error:
                print(f"Error processing batch {batch_start//batch_size + 1}: {str(batch_error)}")
                continue
        
        # Store processed questions if any were successful
        if chapterwise_formatted_questions:
            clean_rephrased_questions[chapter] = chapterwise_formatted_questions
            print(f"\nSuccessfully processed {len(chapterwise_formatted_questions)} questions for chapter {chapter}")
        else:
            print(f"\nWarning: No questions were successfully processed for chapter {chapter}")
        
        return clean_rephrased_questions
        
    except Exception as e:
        print(f"Failed to clean and rephrase questions. Exception Occurred: {type(e).__name__} – {str(e)}")
        print(f"Traceback:\n{traceback.format_exc()}")
        return {}

In [9]:
from typing import Tuple, Dict, Optional
import traceback
import logging

def extract_rephrase_questions(conf_data: dict, topics: list) -> Tuple[Optional[Dict], Optional[Dict]]:
    try:
        # Validate configuration data
        required_fields = ["subject", "grade", "chapter_name"]
        missing_fields = [field for field in required_fields if field not in conf_data]
        if missing_fields:
            raise ValueError(f"Missing required configuration fields: {', '.join(missing_fields)}")
            
        if not topics:
            raise ValueError("No topics provided for question rephrasing")
            
        print("Starting question extraction and rephrasing pipeline...")
        
        # Step 1: Scrape text from PDF/LaTeX
        print("\nStep 1: Scraping textbook content...")
        pdf_text = read_data_from_latex(publication=conf_data['publication'] , chapter_name=conf_data['chapter_name'] ,grade=conf_data['grade'])
        if not pdf_text:
            raise ValueError("No text content extracted from textbook")
        print(f"Successfully extracted {len(pdf_text)} text segments")
            
        # Step 2: Extract questions from text
        print("\nStep 2: Extracting questions from text...")
        extracted_questions = extract_questions(
            subject=conf_data["subject"],
            grade=conf_data["grade"],
            chapter_name=conf_data["chapter_name"],
            chapter_texts=pdf_text.split("\n")
        )
        if not extracted_questions:
            raise ValueError("No questions were extracted from the text")
        print(f"Successfully extracted questions")
            
        # Step 3: Clean and rephrase questions
        print("\nStep 3: Cleaning and rephrasing questions...")
        final_rephrased_questions = rephrase_questions(
            conf_data=conf_data,
            all_questions=extracted_questions,
            topics=topics
        )
        if not final_rephrased_questions:
            raise ValueError("No questions were successfully rephrased")
        print("Successfully rephrased questions")
            
        # Return results
        print("\nPipeline completed successfully!")
        return extracted_questions, final_rephrased_questions
        
    except ValueError as val_err:
        print(f"Validation error: {str(val_err)}")
        print(f"Traceback:\n{traceback.format_exc()}")
        return None, None
        
    except Exception as e:
        print(f"Unexpected error in question processing pipeline: {str(e)}")
        print(f"Exception type: {type(e).__name__}")
        print(f"Traceback:\n{traceback.format_exc()}")
        return None, None
        
    finally:
        print("\nQuestion processing pipeline finished")

In [10]:
# get chapter and book data from defaultConf
with open("defaultConf.json", "r") as f:
    conf_data = json.load(f)

chapter = conf_data["chapter_name"]
subject = conf_data["subject"]


print("Chapter to work on: ", chapter)
print(subject)
print(conf_data['publication'])

Chapter to work on:  Polynomials
mathematics
rsagarwal


In [11]:
import pandas as pd

def process_filename(filename):
    """
    Process filename by stripping whitespace and converting to lowercase
    """
    return filename.strip().lower()
grade = conf_data['grade']
# Match subject to determine sheet name
match subject:
    case "mathematics":
        sheet_name = f"G{grade} Maths"
    case "biology":
        sheet_name = f"G{grade} Science"
    case "physics":
        sheet_name = f"G{grade} Science"
    case "chemistry":
        sheet_name = f"G{grade} Science"
print(sheet_name)
# Read Excel file and process the filename
file_name = process_filename("LEAP Course Creation - Topic LU Prerequisite Misconceptions.xlsx")
misconceptions_df = pd.read_excel(file_name, sheet_name=sheet_name)

# Clean column names
misconceptions_df.columns = misconceptions_df.columns.str.strip()
misconceptions_df.fillna("", inplace=True)

# Process chapter name from conf_data for comparison
chapter_name = conf_data["chapter_name"].strip().lower()

# Get unique topics for the specified chapter
tempTopics = list(misconceptions_df[
    misconceptions_df["Chapter (NCERT/AcadAlly's name)"].str.strip().str.lower() == chapter_name
]["Topic"].unique())

# Create topics dictionary with processed IDs
topics = {}
for i in range(len(tempTopics)):
    topic_id = f"{conf_data['grade']}_{conf_data['subject']}_{chapter_name}_{i+1}"
    topics[topic_id] = tempTopics[i].strip()

print("Topics with ID: ", topics, "\n\n", topics.values())

# Process misconceptions and LUs
total_misconceptions = {}
total_LUs = {}

for i, row in misconceptions_df.iterrows():
    topic = str(row["Topic"]).strip()
    
    if topic in topics.values():
        # Process LUs
        if topic in total_LUs:
            total_LUs[topic].append(row["LUs Covered"])
        else:
            total_LUs[topic] = [row["LUs Covered"]]

        # # Process misconceptions
        # temp_misconceptions = [
        #     row[f"Misconceptions {i}"] 
        #     for i in range(1, 11) 
        #     if row[f"Misconceptions {i}"] != ""
        # ]

        # if topic in total_misconceptions:
        #     total_misconceptions[topic].extend(temp_misconceptions)
        # else:
        #     total_misconceptions[topic] = temp_misconceptions

print(total_LUs, sep="\n\n-------------------------------------------------------------------\n\n")

G10 Maths
Topics with ID:  {'10_mathematics_polynomials_1': 'Geometrical Interpretation of the Zeros of Linear and Quadratic Polynomials', '10_mathematics_polynomials_2': 'Geometrical Interpretation of the Zeros of Polynomials of Degree Greater Than 2', '10_mathematics_polynomials_3': 'Relationship between zeros and coefficients of a polynomial', '10_mathematics_polynomials_4': 'Formation of polynomial from given sum and product of zeroes'} 

 dict_values(['Geometrical Interpretation of the Zeros of Linear and Quadratic Polynomials', 'Geometrical Interpretation of the Zeros of Polynomials of Degree Greater Than 2', 'Relationship between zeros and coefficients of a polynomial', 'Formation of polynomial from given sum and product of zeroes'])
{'Geometrical Interpretation of the Zeros of Linear and Quadratic Polynomials': ['Geometrical meaning of zero of a linear polynomial', 'Geometrical meaning of the zeroes of a quadratic polynomial'], 'Geometrical Interpretation of the Zeros of Polyno

C:\Users\aniket singh\AppData\Local\Temp\ipykernel_12884\609127329.py:26: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  misconceptions_df.fillna("", inplace=True)


In [12]:
total_LUs

{'Geometrical Interpretation of the Zeros of Linear and Quadratic Polynomials': ['Geometrical meaning of zero of a linear polynomial',
  'Geometrical meaning of the zeroes of a quadratic polynomial'],
 'Geometrical Interpretation of the Zeros of Polynomials of Degree Greater Than 2': ['Geometrical meaning of the zeroes of a cubic polynomial',
  'Geometrical meaning of the zeroes of a polynomial p(x) of degree n (n>3)'],
 'Relationship between zeros and coefficients of a polynomial': ['Relationship between zero and coefficients of a linear polynomial',
  'Relationship between zeroes and coefficients of a quadratic polynomial',
  'Relationship between zeroes and coefficients of a cubic polynomial'],
 'Formation of polynomial from given sum and product of zeroes': ['Formation of quadratic polynomial from given sum and product of zeroes']}

In [13]:
extracted_raw_questions_json ,clean_rephrased_questions_json = extract_rephrase_questions(conf_data, topics)


Starting question extraction and rephrasing pipeline...

Step 1: Scraping textbook content...
Successfully processed LaTeX file from zip: C:\Users\ANIKET~1\AppData\Local\Temp\tmpk6ie5v3y\2025_02_04_3785c2277acb8114accbg\2025_02_04_3785c2277acb8114accbg.tex
Successfully extracted 56030 text segments

Step 2: Extracting questions from text...
Processing chunk 1/6
Processing chunk 2/6
Error decoding JSON response: JSONDecodeError - Invalid \escape: line 1 column 120 (char 119)

 Still trying to work on particular exceptions ...
Before removing \ issue:  {
    "questions": [
        {
            "question": "\begin{aligned} \text{Find the zeros of the quadratic polynomial} \quad x^{2}+7x+12 \quad \text{and verify the relationship between the zeros and the coefficients.} \end{aligned}"
        },
        {
            "question": "\begin{aligned} \text{Find the zeros of the quadratic polynomial} \quad x^{2}-2x-8 \quad \text{and verify the relationship between the zeros and the coefficients

In [14]:
print("Chapter name: ",chapter, ",  Raw questions: ",  len(extracted_raw_questions_json))
print("Chapter name: ",chapter, ",  Cleaned questions: ",  len(clean_rephrased_questions_json[chapter]))

Chapter name:  Polynomials ,  Raw questions:  121
Chapter name:  Polynomials ,  Cleaned questions:  121


In [15]:
extracted_raw_questions_json

[{'question': '\\begin{aligned} \\text{Find the zeros of the quadratic polynomial} \\quad x^{2}+7x+12 \\quad \\text{and verify the relationship between the zeros and the coefficients.} \\end{aligned}'},
 {'question': '\\begin{aligned} \\text{Find the zeros of the quadratic polynomial} \\quad x^{2}-2x-8 \\quad \\text{and verify the relationship between the zeros and the coefficients.} \\end{aligned}'},
 {'question': '\\begin{aligned} \\text{Find the zeros of the quadratic polynomial} \\quad x^{2}+3x-10 \\quad \\text{and verify the relationship between the zeros and the coefficients.} \\end{aligned}'},
 {'question': '\\begin{aligned} \\text{Find the zeros of the quadratic polynomial} \\quad 4x^{2}-4x-3 \\quad \\text{and verify the relationship between the zeros and the coefficients.} \\end{aligned}'},
 {'question': '\\begin{aligned} \\text{Find the zeros of the quadratic polynomial} \\quad 5x^{2}-4-8x \\quad \\text{and verify the relationship between the zeros and the coefficients.} \\en

In [16]:
# extracted_raw_questions_json
for json in clean_rephrased_questions_json[conf_data['chapter_name']]:
    print(json['question'])

\begin{aligned} &\text{Determine the roots of the quadratic polynomial} \quad x^{2}+5x+6 \ \quad &\text{and confirm the relationship between the roots and the coefficients.} \end{aligned}
\begin{aligned} &\text{Identify the zeros of the quadratic polynomial} \quad x^{2}-3x-10 \ \quad &\text{and verify the relationship between the zeros and the coefficients.} \end{aligned}
\begin{aligned} &\text{Calculate the zeros of the quadratic polynomial} \quad x^{2}+4x-12 \ \quad &\text{and verify the relationship between the zeros and the coefficients.} \end{aligned}
\begin{aligned} &\text{Find the roots of the quadratic polynomial} \quad 3x^{2}-3x-2 \ \quad &\text{and confirm the relationship between the roots and the coefficients.} \end{aligned}
\begin{aligned} &\text{Determine the zeros of the quadratic polynomial} \quad 6x^{2}-5-9x \ \quad &\text{and verify the relationship between the zeros and the coefficients.} \end{aligned}
\begin{aligned} &\text{What are the zeros of the quadratic polyno

# Solution and Distractors(Based on misconceptions)

In [17]:
llm = ChatOpenAI(model="gpt-4o", temperature=0.2, api_key = OPENAI_API_KEY ,  request_timeout=30.0)

In [18]:
questions_df = pd.DataFrame(clean_rephrased_questions_json[conf_data["chapter_name"]])
questions_df

,topic,topic_id,question
0,Relationship between zeros and coefficients of...,10_mathematics_polynomials_3,\begin{aligned} &\text{Determine the roots of ...
1,Relationship between zeros and coefficients of...,10_mathematics_polynomials_3,\begin{aligned} &\text{Identify the zeros of t...
2,Relationship between zeros and coefficients of...,10_mathematics_polynomials_3,\begin{aligned} &\text{Calculate the zeros of ...
3,Relationship between zeros and coefficients of...,10_mathematics_polynomials_3,\begin{aligned} &\text{Find the roots of the q...
4,Relationship between zeros and coefficients of...,10_mathematics_polynomials_3,\begin{aligned} &\text{Determine the zeros of ...
...,...,...,...
116,Geometrical Interpretation of the Zeros of Pol...,10_mathematics_polynomials_2,\begin{aligned} &\text{Which of the following ...
117,Relationship between zeros and coefficients of...,10_mathematics_polynomials_3,\begin{aligned} &\text{If 3 is one of the zero...
118,Geometrical Interpretation of the Zeros of Pol...,10_mathematics_polynomials_2,\begin{aligned} &\text{Given that } \sqrt{3} \...
119,Geometrical Interpretation of the Zeros of Pol...,10_mathematics_polynomials_2,\begin{aligned} &\text{What is the quotient wh...


In [19]:
import json
def parse_Response(response):
    if isinstance(response, str):
        # print(response)
        start_index = response.find("{")
        end_index = response.rfind("}")
        if start_index != -1 and end_index != -1:
            valid_json_content = response[start_index : end_index + 1]
            try:
                JSON_response = json.loads(valid_json_content.replace("\n", "").replace("\(", "").replace("\)", ""))
                # append_list_to_file(JSON_response)
                return JSON_response
            except json.JSONDecodeError as e:
                print(f"Error decoding JSON response: {e.__class__.__name__} - {e}\n\n Still trying to work on particular exceptions ...")
                if str(e).startswith("Extra data"):
                    json_parts = valid_json_content.split('}\n{')
                    if json_parts:
                        first_json = json_parts[0] + '}'
                        return parse_Response(first_json)  
                elif str(e).startswith("Invalid \escape"):
                    print("Before removing \\ issue: ", valid_json_content)
                    str1 = valid_json_content.replace("\\\\\\\\\\", "fvback")
                    str1 = str1.replace("\\\\\\\\", "frback")
                    str1 = str1.replace("\\\\\\", "trlback")
                    str1 = str1.replace("\\\\", "dblback")
                    str1 = str1.replace("\\", "\\\\")
                    str1 = str1.replace("fvback","\\\\\\\\\\")
                    str1 = str1.replace("frback", "\\\\\\\\")
                    str1 = str1.replace("trlback","\\\\\\")
                    strfinal = str1.replace("dblback","\\\\")
                    print("After removing \\ issue: ", strfinal)
                    JSON_response = json.loads(strfinal.replace("\n", ""))
                    return JSON_response
                else:
                    print("Actual Content: ", valid_json_content)
        else:
            print("No valid JSON content found in the response.")
        # time.sleep(50)
    elif isinstance(response, dict):
        return response
    else:
        print("No response message found", type(response))

<>:10: SyntaxWarning: invalid escape sequence '\('
<>:10: SyntaxWarning: invalid escape sequence '\)'
<>:20: SyntaxWarning: invalid escape sequence '\e'
<>:10: SyntaxWarning: invalid escape sequence '\('
<>:10: SyntaxWarning: invalid escape sequence '\)'
<>:20: SyntaxWarning: invalid escape sequence '\e'
C:\Users\aniket singh\AppData\Local\Temp\ipykernel_12884\3339088456.py:10: SyntaxWarning: invalid escape sequence '\('
  JSON_response = json.loads(valid_json_content.replace("\n", "").replace("\(", "").replace("\)", ""))
C:\Users\aniket singh\AppData\Local\Temp\ipykernel_12884\3339088456.py:10: SyntaxWarning: invalid escape sequence '\)'
  JSON_response = json.loads(valid_json_content.replace("\n", "").replace("\(", "").replace("\)", ""))
C:\Users\aniket singh\AppData\Local\Temp\ipykernel_12884\3339088456.py:20: SyntaxWarning: invalid escape sequence '\e'
  elif str(e).startswith("Invalid \escape"):


In [20]:
## Generating solutions
prompt_template = PromptTemplate(
    input_variables=["system_prompt", "question", "topic", "topic_id", "lulist"],
    template="{system_prompt}\n\n\nGenerate a hint, and solution for the given question. Also rephrase the given question according to the specified steps.\nHere are the required context:\nQuestion: {question}\nLearning unit data which defines scope from which solution should be generated: {lulist}\nTopic: {topic}\nTopic ID: {topic_id}\n\nReturn the output in the specified JSON format."
)

chain = LLMChain(llm=llm, prompt=prompt_template)
# chain = prompt_template | llm
# Function to process each question and generate hint, solution, and final answer
def generate_explanation(subject, question, topic, topic_id):
    try:
        match subject:
            case "mathematics":
                system = MATHEMATICS_SOLUTION_SYSTEM_PROMPT
            case "biology":
                system = BIOLOGY_SOLUTION_SYSTEM_PROMPT
            case "physics":
                system = PHYSICS_SOLUTION_SYSTEM_PROMPT
            case "chemistry":
                system = CHEMISTRY_SOLUTION_SYSTEM_PROMPT

        response = chain.run(
            system_prompt=system,
            lulist=total_LUs,
            question=question,
            topic=topic,
            topic_id=topic_id
        )
        return response
    except Exception as e:
        print(f"Error processing question: {question}\nError: {str(e)}")
        return None
# parser = PydanticOutputParser(pydantic_object=sol_data)
# global explanations
explanations = []
def loopForSolution(subject, questions_df):
    counter = 0
    retries_counter = 0
    maxCounter = len(questions_df)
    while counter < maxCounter:
        row = questions_df.iloc[counter]
        question = row['question']
        topic = row['topic']
        topic_id = row['topic_id']
        print(f"Processing question {counter + 1} out of {len(questions_df)}")
        explanation = generate_explanation(subject, question, topic, topic_id)
        parsed_explanation = parse_Response(explanation)
        if parsed_explanation:
            parsed_explanation["question"] = question
            parsed_explanation["topic"] = topic
            parsed_explanation["topic_id"] = topic_id
            explanations.append(parsed_explanation)
            counter += 1
            retries_counter = 0
        else:
            print("Issue in openai response", explanation, "\n\n Inputs: \n\n", question, "\n\n Topic: ", topic)
            print(f"Failed to parse explanation or generate solution for question {counter + 1}, retrying...")
            retries_counter += 1
        if retries_counter == 3:
            counter += 1

loopForSolution(subject, questions_df)

C:\Users\aniket singh\AppData\Local\Temp\ipykernel_12884\1295566236.py:7: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=llm, prompt=prompt_template)
C:\Users\aniket singh\AppData\Local\Temp\ipykernel_12884\1295566236.py:22: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = chain.run(


Processing question 1 out of 121
Processing question 2 out of 121
Processing question 3 out of 121
Processing question 4 out of 121
Processing question 5 out of 121
Processing question 6 out of 121
Processing question 7 out of 121
Processing question 8 out of 121
Processing question 9 out of 121
Processing question 10 out of 121
Processing question 11 out of 121
Processing question 12 out of 121
Processing question 13 out of 121
Processing question 14 out of 121
Processing question 15 out of 121
Processing question 16 out of 121
Processing question 17 out of 121
Processing question 18 out of 121
Processing question 19 out of 121
Processing question 20 out of 121
Processing question 21 out of 121
Processing question 22 out of 121
Processing question 23 out of 121
Processing question 24 out of 121
Processing question 25 out of 121
Processing question 26 out of 121
Processing question 27 out of 121
Processing question 28 out of 121
Processing question 29 out of 121
Processing question 30 

In [21]:
explanations

[{'hint': '\\begin{aligned} &\\text{Use the quadratic formula to find the roots and verify } \\alpha + \\beta = -b/a, \\alpha \\beta = c/a \\end{aligned}',
  'solution': '\\begin{aligned} &x^2 + 5x + 6 = 0 \\\\ &\\implies \\quad x = \\frac{-b \\pm \\sqrt{b^2 - 4ac}}{2a} \\\\ &\\implies \\quad x = \\frac{-5 \\pm \\sqrt{5^2 - 4 \\cdot 1 \\cdot 6}}{2 \\cdot 1} \\\\ &\\implies \\quad x = \\frac{-5 \\pm \\sqrt{25 - 24}}{2} \\\\ &\\implies \\quad x = \\frac{-5 \\pm 1}{2} \\\\ &\\implies \\quad x = -3, -2 \\\\ &\\text{Roots are } \\alpha = -3, \\beta = -2 \\\\ &\\text{Verify: } \\alpha + \\beta = -3 + (-2) = -5 = -b/a \\\\ &\\alpha \\beta = (-3)(-2) = 6 = c/a \\\\ &\\therefore \\text{Relationship confirmed} \\\\ \\end{aligned}',
  'question': '\\begin{aligned} &\\text{Determine the roots of the quadratic polynomial} \\quad x^{2}+5x+6 \\ \\quad &\\text{and confirm the relationship between the roots and the coefficients.} \\end{aligned}',
  'topic': 'Relationship between zeros and coefficients 

### Generating Distractors

In [22]:
data = pd.DataFrame(explanations)


In [23]:
data['misconception_options'] = ''
data

,hint,solution,question,topic,topic_id,misconception_options
0,\begin{aligned} &\text{Use the quadratic formu...,\begin{aligned} &x^2 + 5x + 6 = 0 \\ &\implies...,\begin{aligned} &\text{Determine the roots of ...,Relationship between zeros and coefficients of...,10_mathematics_polynomials_3,
1,\begin{aligned} &\text{Use the quadratic formu...,\begin{aligned} &\text{Given polynomial: } x^2...,\begin{aligned} &\text{Identify the zeros of t...,Relationship between zeros and coefficients of...,10_mathematics_polynomials_3,
2,\begin{aligned} &\text{Use the quadratic formu...,\begin{aligned} &x^2 + 4x - 12 = 0 \\ &\implie...,\begin{aligned} &\text{Calculate the zeros of ...,Relationship between zeros and coefficients of...,10_mathematics_polynomials_3,
3,\begin{aligned} &\text{Use the quadratic formu...,\begin{aligned} &\text{Given polynomial: } 3x^...,\begin{aligned} &\text{Find the roots of the q...,Relationship between zeros and coefficients of...,10_mathematics_polynomials_3,
4,\begin{aligned} &\text{Use the quadratic formu...,\begin{aligned} &\text{Given polynomial: } 6x^...,\begin{aligned} &\text{Determine the zeros of ...,Relationship between zeros and coefficients of...,10_mathematics_polynomials_3,
...,...,...,...,...,...,...
116,\begin{aligned} &\text{Analyze the discriminan...,\begin{aligned} &f(x) = x^4 + 5x^2 + 7 \\ &\te...,\begin{aligned} &\text{Which of the following ...,Geometrical Interpretation of the Zeros of Pol...,10_mathematics_polynomials_2,
117,\begin{aligned} &\text{Use the factor theorem ...,\begin{aligned} &p(x) = x^3 - 7x^2 + 13x - 6 \...,\begin{aligned} &\text{If 3 is one of the zero...,Relationship between zeros and coefficients of...,10_mathematics_polynomials_3,
118,\begin{aligned} &\text{Use the fact that the s...,\begin{aligned} &\text{Given zeros: } \sqrt{3}...,\begin{aligned} &\text{Given that } \sqrt{3} \...,Geometrical Interpretation of the Zeros of Pol...,10_mathematics_polynomials_2,
119,\begin{aligned} &\text{Use polynomial long div...,\begin{aligned} &\text{Divide } 4x^4 + 6x^3 - ...,\begin{aligned} &\text{What is the quotient wh...,Geometrical Interpretation of the Zeros of Pol...,10_mathematics_polynomials_2,


In [24]:
for i in range(len(data)):
    print(data['solution'][i].split("\n"))

['\\begin{aligned} &x^2 + 5x + 6 = 0 \\\\ &\\implies \\quad x = \\frac{-b \\pm \\sqrt{b^2 - 4ac}}{2a} \\\\ &\\implies \\quad x = \\frac{-5 \\pm \\sqrt{5^2 - 4 \\cdot 1 \\cdot 6}}{2 \\cdot 1} \\\\ &\\implies \\quad x = \\frac{-5 \\pm \\sqrt{25 - 24}}{2} \\\\ &\\implies \\quad x = \\frac{-5 \\pm 1}{2} \\\\ &\\implies \\quad x = -3, -2 \\\\ &\\text{Roots are } \\alpha = -3, \\beta = -2 \\\\ &\\text{Verify: } \\alpha + \\beta = -3 + (-2) = -5 = -b/a \\\\ &\\alpha \\beta = (-3)(-2) = 6 = c/a \\\\ &\\therefore \\text{Relationship confirmed} \\\\ \\end{aligned}']
['\\begin{aligned} &\\text{Given polynomial: } x^2 - 3x - 10 \\\\ &\\text{Quadratic formula: } x = \\frac{-b \\pm \\sqrt{b^2 - 4ac}}{2a} \\\\ &a = 1, \\quad b = -3, \\quad c = -10 \\\\ &\\implies x = \\frac{-(-3) \\pm \\sqrt{(-3)^2 - 4 \\cdot 1 \\cdot (-10)}}{2 \\cdot 1} \\\\ &\\implies x = \\frac{3 \\pm \\sqrt{9 + 40}}{2} \\\\ &\\implies x = \\frac{3 \\pm \\sqrt{49}}{2} \\\\ &\\implies x = \\frac{3 \\pm 7}{2} \\\\ &\\implies x = 5 \

In [25]:
# Define the prompt template for generating misconceptions
prompt_template = PromptTemplate(
    input_variables=["system_prompt","topic", "topic_id", "question", "hint", "solution"],
    template="{system_prompt}\n\nGenerate one correct option and appropriate incorrect options in latex using given solution, hint, and misconceptions for the given question.\n\nTopic ID: {topic_id}\nTopic: {topic}\nQuestion: {question}\nHint: {hint}\nSolution: {solution}\n\nReturn the output in the specified JSON format."
)

# Extract relevant information from the 'explanation' column

def extract_explanation_details(explanation):
    try:
        explanation_data = json.loads(explanation)
        hint = explanation_data.get('hint', 'No hint available')
        solution = explanation_data.get('solution', 'No solution available')
        return hint, solution
    except json.JSONDecodeError:
        return 'No hint available', 'No solution available'

# Function to generate misconceptions-based incorrect options

def generate_incorrect_options(row):

    chain = LLMChain(llm=llm, prompt=prompt_template)
    try:
        response = chain.run(
            system_prompt=DISTRACTOR_SYSTEM_PROMPT,
            topic=row['topic'],
            topic_id=row['topic_id'],
            question=row['question'],
            hint=row['hint'],
            solution=row['solution']
        )
        # if response.strip().startswith("```json"):
        #     response = response.strip().strip("```json").strip("```").strip()
        return response
    except Exception as e:
        print(f"Unexpected error: {str(e)}")
        return None

# Apply the function to generate misconceptions-based incorrect options for each question
misconception_options = []
def loopForDistractor(data):
    distractor_counter = 0
    retries_counter = 0
    maxCounter = len(data)
    while distractor_counter<maxCounter:
        row = data.loc[distractor_counter]
        print(f"Processing question {distractor_counter + 1} out of {len(data)}")
        misconception_option = generate_incorrect_options(row)
        parsed_response = parse_Response(misconception_option)

        if parsed_response:
            # misconception_options.append(parsed_response)
            # row['misconception_options'] = parsed_response
            # print(distractor_counter)
            # print(data.loc[distractor_counter , 'misconception_options'])
            # print(parsed_response)

            data.loc[distractor_counter , 'misconception_options'] =[parsed_response]
            distractor_counter += 1
            retries_counter = 0
        else:
            print(f"Failed to parse response or generate distractors for question {distractor_counter + 1}, retrying...")
            retries_counter += 1
        if retries_counter == 3:
            # misconception_option.append("")
            data.loc[distractor_counter , 'misconception_options'] = ""
            distractor_counter += 1

loopForDistractor(data)
# data['misconception_options'] = misconception_options

Processing question 1 out of 121
Processing question 2 out of 121
Processing question 3 out of 121
Processing question 4 out of 121
Processing question 5 out of 121
Processing question 6 out of 121
Processing question 7 out of 121
Processing question 8 out of 121
Processing question 9 out of 121
Processing question 10 out of 121
Processing question 11 out of 121
Processing question 12 out of 121
Processing question 13 out of 121
Processing question 14 out of 121
Processing question 15 out of 121
Processing question 16 out of 121
Unexpected error: Request timed out.
No response message found <class 'NoneType'>
Failed to parse response or generate distractors for question 16, retrying...
Processing question 16 out of 121
Processing question 17 out of 121
Processing question 18 out of 121
Processing question 19 out of 121
Processing question 20 out of 121
Processing question 21 out of 121
Processing question 22 out of 121
Processing question 23 out of 121
Processing question 24 out of 121

In [26]:
for i in range(len(data)):
    print(data['misconception_options'][i])

[{'correct_option': '\\begin{aligned} &\\alpha = -3, \\beta = -2 \\end{aligned}', 'option1': {'option': '\\begin{aligned} &\\alpha = -2, \\beta = -3 \\end{aligned}', 'rationale': '\\begin{aligned} &\\text{Confused the order of the roots, which does not affect the relationship.} \\end{aligned}'}, 'option2': {'option': '\\begin{aligned} &\\alpha = 3, \\beta = 2 \\end{aligned}', 'rationale': '\\begin{aligned} &\\text{Forgot to apply the negative sign from the quadratic formula correctly.} \\end{aligned}'}, 'option3': {'option': '\\begin{aligned} &\\alpha = -1, \\beta = -6 \\end{aligned}', 'rationale': '\\begin{aligned} &\\text{Ignored the correct calculation of the discriminant.} \\end{aligned}'}}]
[{'correct_option': '\\begin{aligned} &5 \\quad \\text{and} \\quad -2 \\end{aligned}', 'option1': {'option': '\\begin{aligned} &3 \\quad \\text{and} \\quad -5 \\end{aligned}', 'rationale': '\\begin{aligned} &\\text{Confused the signs in the quadratic formula, leading to incorrect zeros.} \\end{

### Output Formatting & Pushing into the DB

In [27]:
# Function to process each row and extract required fields
def process_row(row, index):
    try:
        misconception_data = row["misconception_options"]
        # Extract topic, question, and solution details
        topic_id = row["topic_id"]
        topic_name = row["topic"]
        question = row["question"]
        hint = row["hint"]
        explanation = row["solution"]
        # correct_answer = row["final_answer"]
        chapter = conf_data['chapter_name']

        # Extract option and rationale details
        options = {}
        for i in range(1, 4):  # Assuming there are 4 options for each question
            options["correct_answer"] = misconception_data[0]["correct_option"]
            option_key = f'option{i}'
            option_info = misconception_data[0][option_key]
            options[f'option{i}'] = option_info.get('option', '')
            options[f'dr{i}'] = option_info.get('rationale', '')
        
        # Construct the final JSON structure
        start = "\\begin{aligned}"
        end = "\\end{aligned}"
        
        combined_data = {
            'topic_id': topic_id,
            'topic_name': topic_name,
            'grade' : conf_data['grade'],
            'board' : "CBSE",
            
            'subject' : subject,
            'chapter_name' : chapter,
            'publication' : conf_data['publication'],
            'question': [{"content": question}],
            # if (str(question).startswith("\\begin{aligned}") and str(question).endswith("\\end{aligned}")) else [{"content": start + question + end}],

            'hint': [{"content": hint}],
            # if (str(hint).startswith("\\begin{aligned}") and str(hint).endswith("\\end{aligned}")) else [{"content": start + hint + end}],

            'solution': [{"content": explanation}],
            # if (str(explanation).startswith("\\begin{aligned}") and str(explanation).endswith("\\end{aligned}")) else [{"content": start + explanation + end}],

            'final_answer': [{"content": options.get('correct_answer', '')}],
            # if (str(options.get('correct_answer', '')).startswith("\\begin{aligned}") and str(options.get('correct_answer', '')).endswith("\\end{aligned}")) else [{"content": start + options.get('correct_answer', '') + end}],

            'option1': [{"content": options.get('option1', '')}],
            # if (str(options.get('option1')).startswith("\\begin{aligned}") and str(options.get('option1')).endswith("\\end{aligned}")) else [{"content": start + str(options.get('option1', '')) + end} ],
            'dr1': [{"content": options.get('dr1', '')}],
            'option2': [{"content": options.get('option2', '')}],
            # if (str(options.get('option2')).startswith("\\begin{aligned}") and str(options.get('option2')).endswith("\\end{aligned}")) else [{"content": start + str(options.get('option2', '')) + end}],
            'dr2': [{"content": options.get('dr2', '')}],
            'option3': [{"content": options.get('option3', '')}],
            # if (str(options.get('option3')).startswith("\\begin{aligned}") and str(options.get('option3')).endswith("\\end{aligned}")) else [{"content": start + str(options.get('option3', '')) + end}],
            'dr3': [{"content": options.get('dr3', '')}]


        }

        return combined_data
    except Exception as e:
        print(f"Exception Occurred: {type(e).__name__} – {e} \n {traceback.format_exc()}")


# Process all rows in the dataframe
final_output = [process_row(row, idx) for idx, row in data.iterrows()]
final_output[2]

{'topic_id': '10_mathematics_polynomials_3',
 'topic_name': 'Relationship between zeros and coefficients of a polynomial',
 'grade': '10',
 'board': 'CBSE',
 'subject': 'mathematics',
 'chapter_name': 'Polynomials',
 'publication': 'rsagarwal',
 'question': [{'content': '\\begin{aligned} &\\text{Calculate the zeros of the quadratic polynomial} \\quad x^{2}+4x-12 \\ \\quad &\\text{and verify the relationship between the zeros and the coefficients.} \\end{aligned}'}],
 'hint': [{'content': '\\begin{aligned} &\\text{Use the quadratic formula to find zeros and verify using } \\alpha + \\beta = -\\frac{b}{a}, \\alpha \\beta = \\frac{c}{a} \\end{aligned}'}],
 'solution': [{'content': '\\begin{aligned} &x^2 + 4x - 12 = 0 \\\\ &\\implies \\quad x = \\frac{-b \\pm \\sqrt{b^2 - 4ac}}{2a} \\\\ &\\implies \\quad x = \\frac{-4 \\pm \\sqrt{4^2 - 4 \\cdot 1 \\cdot (-12)}}{2 \\cdot 1} \\\\ &\\implies \\quad x = \\frac{-4 \\pm \\sqrt{16 + 48}}{2} \\\\ &\\implies \\quad x = \\frac{-4 \\pm \\sqrt{64}}{2}

In [28]:
final_output

[{'topic_id': '10_mathematics_polynomials_3',
  'topic_name': 'Relationship between zeros and coefficients of a polynomial',
  'grade': '10',
  'board': 'CBSE',
  'subject': 'mathematics',
  'chapter_name': 'Polynomials',
  'publication': 'rsagarwal',
  'question': [{'content': '\\begin{aligned} &\\text{Determine the roots of the quadratic polynomial} \\quad x^{2}+5x+6 \\ \\quad &\\text{and confirm the relationship between the roots and the coefficients.} \\end{aligned}'}],
  'hint': [{'content': '\\begin{aligned} &\\text{Use the quadratic formula to find the roots and verify } \\alpha + \\beta = -b/a, \\alpha \\beta = c/a \\end{aligned}'}],
  'solution': [{'content': '\\begin{aligned} &x^2 + 5x + 6 = 0 \\\\ &\\implies \\quad x = \\frac{-b \\pm \\sqrt{b^2 - 4ac}}{2a} \\\\ &\\implies \\quad x = \\frac{-5 \\pm \\sqrt{5^2 - 4 \\cdot 1 \\cdot 6}}{2 \\cdot 1} \\\\ &\\implies \\quad x = \\frac{-5 \\pm \\sqrt{25 - 24}}{2} \\\\ &\\implies \\quad x = \\frac{-5 \\pm 1}{2} \\\\ &\\implies \\quad 

In [ ]:
sadsafsd

#### Latex Formatting to store in DB

In [29]:
from config import MONGO_URL , DB_NAME

In [30]:
dbCollection = "LatexTest"
client = AsyncIOMotorClient(MONGO_URL)

In [31]:
async def pushToDB(dbContent, subject):
    try:
        db = client[DB_NAME]
        que_collection = db[dbCollection]

        dbContent["status"] = "not_reviewed"
        dbContent["comment"] = ""
        dbContent["testFlag"] = "true"
        dbContent["subject"] = subject
        await que_collection.insert_one(dbContent)
        print("Data Uploaded to the database successfully.")
    except Exception as e:
        print("Exception Occurred: ", type(e).__name__, "–", e, "\n", traceback.format_exc())

In [32]:
for doc in final_output:
    await pushToDB(doc, subject)
client.close()

Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database su

In [ ]:
with open('defaultConf.json', 'r') as f:
    conf_data = json.load(f)

conf_data

{'publication': 'arihant',
 'grade': '10',
 'subject': 'mathematics',
 'term': '1',
 'chapter_name': 'Pair of Linear Equations in Two Variables'}

In [ ]:
dbCollection = "LatexTest"
client = AsyncIOMotorClient(MONGO_URL)

In [ ]:
async def getFromDB():
    try:
        db = client[DB_NAME]
        que_collection = db[dbCollection]
        res = await que_collection.find({"grade": conf_data['grade'], "subject": conf_data['subject'], "chapter_name": conf_data['chapter_name']}).to_list(length=10000)
        return res
    except Exception as e:
        print("Exception Occurred: ", type(e).__name__, "–", e, "\n", traceback.format_exc())

In [ ]:
questionsList = await getFromDB()
client.close()

In [ ]:
len(questionsList)

187

In [ ]:
questionsList[-1]

{'_id': ObjectId('67a310140b8be3d67104e8ab'),
 'topic_id': '10_mathematics_pair of linear equations in two variables_1',
 'topic_name': 'Introduction of linear pair of equations',
 'grade': '10',
 'board': 'CBSE',
 'subject': 'mathematics',
 'chapter_name': 'Pair of Linear Equations in Two Variables',
 'publication': 'mtg',
 'question': [{'content': '\\begin{aligned} &\\text{Which of the following statements is true about the lines } x=3 \\text{ and } y=4 \\text{?} \\\\ &\\text{(A) They are parallel to the } y\\text{-axis and } x\\text{-axis respectively.} \\\\ &\\text{(B) They will intersect at a point.} \\\\ &\\text{(C) They are the same line.} \\\\ &\\text{(D) They are perpendicular to each other.} \\end{aligned}'}],
 'hint': [{'content': '\\begin{aligned} &\\text{Identify the orientation of the lines } x=3 \\text{ and } y=4 \\text{ on the coordinate plane.} \\\\ &\\text{Determine their relationship based on their orientation.} \\end{aligned}'}],
 'solution': [{'content': '\\begin{a

In [ ]:
from pydantic import BaseModel, Field

class Question(BaseModel):
    question: str = Field(..., description="Question text", examples=["\\begin{{aligned}} \\text{{Solve given equation}} \\quad x^2 - 4x + 7 = 0. \\end{{aligned}}"])
    qid: int = Field(..., description="Question ID same as provided in input", examples=[224, 8623, 10234])


class SingleCluster(BaseModel):
    cluster1: list[Question]

class Clusters(BaseModel):
    clusters: list[SingleCluster]

Clusters.model_json_schema()

{'$defs': {'Question': {'properties': {'question': {'description': 'Question text',
     'examples': ['\\begin{{aligned}} \\text{{Solve given equation}} \\quad x^2 - 4x + 7 = 0. \\end{{aligned}}'],
     'title': 'Question',
     'type': 'string'},
    'qid': {'description': 'Question ID same as provided in input',
     'examples': [224, 8623, 10234],
     'title': 'Qid',
     'type': 'integer'}},
   'required': ['question', 'qid'],
   'title': 'Question',
   'type': 'object'},
  'SingleCluster': {'properties': {'cluster1': {'items': {'$ref': '#/$defs/Question'},
     'title': 'Cluster1',
     'type': 'array'}},
   'required': ['cluster1'],
   'title': 'SingleCluster',
   'type': 'object'}},
 'properties': {'clusters': {'items': {'$ref': '#/$defs/SingleCluster'},
   'title': 'Clusters',
   'type': 'array'}},
 'required': ['clusters'],
 'title': 'Clusters',
 'type': 'object'}

In [ ]:
questionTopicDict = {}
for i in questionsList:
    if i['topic_id'] in questionTopicDict:
        questionTopicDict[i['topic_id']].append({"question": i['question'], "qid": i['qid']})
    else:
        questionTopicDict[i['topic_id']] = [{"question": i['question'], "qid": i['qid']}]
questionTopicDict

{'10_mathematics_pair of linear equations in two variables_1': [{'question': [{'content': '\\begin{aligned} &\\text{Which of the following represents a linear equation in two variables?} \\end{aligned}'}],
   'qid': '10317-1'},
  {'question': [{'content': '\\begin{aligned} &\\text{Which equation describes the relationship between the number of bags } x, \\\\ &\\text{and the number of baskets, } y \\text{?} \\end{aligned}'}],
   'qid': '10319-1'},
  {'question': [{'content': '\\begin{aligned} &\\text{Which of these pairs of equations is equivalent to the given pair of equations?} \\\\ \\end{aligned}'}],
   'qid': '10331-1'},
  {'question': [{'content': '\\begin{aligned} &\\text{What are the possible values of } \\quad x \\quad \\text{and} \\quad y \\quad \\text{in the given equation?} \\end{aligned}'}],
   'qid': '10331-2'},
  {'question': [{'content': '\\begin{aligned} &\\text{What do the variables } \\quad x \\quad \\text{and} \\quad y \\quad \\text{symbolize in the context of the equ

In [ ]:
list(questionTopicDict.keys())

['10_mathematics_pair of linear equations in two variables_1',
 '10_mathematics_pair of linear equations in two variables_2',
 '10_mathematics_pair of linear equations in two variables_4',
 '10_mathematics_pair of linear equations in two variables_3']

In [ ]:
len(questionTopicDict['10_mathematics_pair of linear equations in two variables_1'])

66

In [ ]:
system_cluster_prompt = """
You are an academic expert who is responsible for clustering similar questions based on different parameters.

## FOLLOW THESE INSTRUCTIONS TO CLUSTER THE QUESTIONS:
    - Read and Understand Each Question: First, carefully analyze each question. Think about its key concept, context, or content.
    - Identify Common Patterns: Next, consider which questions share similar content/context, whether they are asking about the same learning unit, are related to the same topic, or have same structure.
    - Identify and Group Similar Questions: Based on the identified patterns, identify similar questions.
    - Reflect on Your Choices: After grouping, double-check to ensure that questions in the same group are closely related and that no questions have been misplaced.
    - No question should be lost or repeated in the output.
    - Always return all the given questions clustered into groups in the specified JSON format.

###OUTPUT INSTRUCTIONS###:
- Return same question and qid after grouping. Do not change any question or text, just cluster them according to the above criteria.

Return your output in the following json format:
'''
{
  "clusters": [
    {
      "cluster1": [
        {
          "qid": "qid of the question",
          "question": "whole content of the question"
        },
      ]
    },
  ]
}
'''
"""

user_cluster_prompt = """
Group all the questions into different clusters based on their similarity, content, and context behind the question. You will be provided a list of questions with their qid(i.e. question id).

## Input:
List of questions with their qid/s: {questions_input}

Please follow this thought process:
- Read through each question carefully.
- Identify the context, structure or concept within each question.
- Group the questions that share similar concept, context or structure.
- Reflect on your choices and make sure questions are grouped logically.
- Please reason through each step clearly as you perform the clustering.
"""

In [ ]:
from openai import OpenAI
def generateUsingOpenAI(system, user):
    client = OpenAI()
    #generating response from api call
    response = client.beta.chat.completions.parse(
        model = "o3-mini-2025-01-31",
        messages = [
            {"role":"system", "content": system},
            {"role" :"user", "content": user}
        ],
    )
    return response

In [ ]:
topic_ids = list(questionTopicDict.keys())
topic_ids

['10_mathematics_pair of linear equations in two variables_1',
 '10_mathematics_pair of linear equations in two variables_2',
 '10_mathematics_pair of linear equations in two variables_4',
 '10_mathematics_pair of linear equations in two variables_3']

In [ ]:
totalClustering = {}
topic_ids = list(questionTopicDict.keys())
for topic_id in topic_ids:
    print("Processing topic: ", topic_id)
    user = user_cluster_prompt.format(questions_input=questionTopicDict[topic_id])

    #generating response
    raw_response = generateUsingOpenAI(system_cluster_prompt, user)
    parsed_resp = parse_Response(raw_response.choices[0].message.content)
    if parsed_resp:
        totalClustering[topic_id] = parsed_resp['clusters']
        print("Generated response", raw_response, '\n\n')
    else:
        print("Error in response generation: ", raw_response, '\n\n')

Processing topic:  10_mathematics_pair of linear equations in two variables_1
Generated response ParsedChatCompletion[NoneType](id='chatcmpl-AyAphDIFkUmV0DCVnbMXiw6CWuLEL', choices=[ParsedChoice[NoneType](finish_reason='stop', index=0, logprobs=None, message=ParsedChatCompletionMessage[NoneType](content='{\n  "clusters": [\n    {\n      "cluster1": [\n        {\n          "qid": "10317-1",\n          "question": "\\\\begin{aligned} &\\\\text{Which of the following represents a linear equation in two variables?} \\\\end{aligned}"\n        },\n        {\n          "qid": "10317-2",\n          "question": "\\\\begin{aligned} &\\\\text{Which linear equation represents the condition if Akhila spends } \\\\\\\\ &\\\\text{₹ 20 on rides costing ₹ 3 each and hoopla games costing ₹ 4 each?} \\\\end{aligned}"\n        },\n        {\n          "qid": "10317-3",\n          "question": "\\\\begin{aligned} & \\\\text{Which of the following equations represents the pair of linear equations } 3x - 4y -

In [ ]:
totalClustering

{'10_mathematics_pair of linear equations in two variables_1': [{'cluster1': [{'qid': '10317-1',
     'question': '\\begin{aligned} &\\text{Which of the following represents a linear equation in two variables?} \\end{aligned}'},
    {'qid': '10317-2',
     'question': '\\begin{aligned} &\\text{Which linear equation represents the condition if Akhila spends } \\\\ &\\text{₹ 20 on rides costing ₹ 3 each and hoopla games costing ₹ 4 each?} \\end{aligned}'},
    {'qid': '10317-3',
     'question': '\\begin{aligned} & \\text{Which of the following equations represents the pair of linear equations } 3x - 4y - 10 = 0, 9y + 18 = 6x \\text{?} \\end{aligned}'},
    {'qid': '10317-4',
     'question': '\\begin{aligned} &\\text{Which of the following represents the given linear equation?} \\\\ &217x + 131y = 913 \\end{aligned}'},
    {'qid': '10317-5',
     'question': '\\begin{aligned} &\\text{Which of the following represents the equation of the line } \\quad k_{5} \\text{?} \\end{aligned}'},
  

In [ ]:
count = 0
for i in list(totalClustering.keys()):
    for j in range(len(totalClustering[i])):
        # print("\n\nSingle doc ======================>",totalClustering[i][j][list(totalClustering[i][j].keys())[0]])
        count += len(totalClustering[i][j][list(totalClustering[i][j].keys())[0]])
count

183

In [ ]:
dbCollection = "LatexTest"
client = AsyncIOMotorClient(MONGO_URL)
async def updateQIDs(firstqid, totalClustering):
    try:
        db = client[DB_NAME]
        que_collection = db[dbCollection]
        for topic_id in list(totalClustering.keys()):
            for j in range(len(totalClustering[topic_id])):
                print("Inside cluster: ", totalClustering[topic_id][j][list(totalClustering[topic_id][j].keys())[0]])
                firstqid = int(totalClustering[topic_id][j][list(totalClustering[topic_id][j].keys())[0]][0]["qid"])
                print(firstqid, type(firstqid))
                for k in range(len(totalClustering[topic_id][j][list(totalClustering[topic_id][j].keys())[0]])):
                    firstqid += 1
                    print("Later qid: ",totalClustering[topic_id][j][list(totalClustering[topic_id][j].keys())[0]][k]["qid"], "------", f'{firstqid}-{k+1}')
                    await que_collection.update_one({"topic_id": topic_id, "qid": totalClustering[topic_id][j][list(totalClustering[topic_id][j].keys())[0]][k]["qid"]}, {"$set": {"qid": f'{firstqid}-{k+1}'}})
                
        print("Clustering updated successfully.")
    except Exception as e:
        print("Exception Occurred: ", type(e).__name__, "–", e, "\n", traceback.format_exc())

In [ ]:
await updateQIDs(totalClustering)
client.close()

Inside cluster:  [{'qid': '10317-1', 'question': '\\begin{aligned} &\\text{Which of the following represents a linear equation in two variables?} \\end{aligned}'}, {'qid': '10317-2', 'question': '\\begin{aligned} &\\text{Which linear equation represents the condition if Akhila spends } \\\\ &\\text{₹ 20 on rides costing ₹ 3 each and hoopla games costing ₹ 4 each?} \\end{aligned}'}, {'qid': '10317-3', 'question': '\\begin{aligned} & \\text{Which of the following equations represents the pair of linear equations } 3x - 4y - 10 = 0, 9y + 18 = 6x \\text{?} \\end{aligned}'}, {'qid': '10317-4', 'question': '\\begin{aligned} &\\text{Which of the following represents the given linear equation?} \\\\ &217x + 131y = 913 \\end{aligned}'}, {'qid': '10317-5', 'question': '\\begin{aligned} &\\text{Which of the following represents the equation of the line } \\quad k_{5} \\text{?} \\end{aligned}'}, {'qid': '10317-6', 'question': '\\begin{aligned} &\\text{If the monthly fixed charges are } ₹ a \\text{